# ⚡ Module 4.4: TD Learning & SARSA for Image Processing

## Bootstrapping: Learning Before the Episode Ends

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_04_Basic_RL_Algorithms/04_TD_Learning_SARSA/td_learning_sarsa.ipynb)

---

## 🎯 Learning Objectives
1. Derive the **TD(0) update** rule from the MC update by introducing bootstrapping
2. Understand the **bias-variance tradeoff** between MC and TD methods
3. Implement **n-step TD** methods and understand the spectrum from TD(0) to MC
4. Derive and implement **SARSA** (on-policy TD control)
5. Derive **TD(λ)** with eligibility traces and prove forward-backward equivalence
6. Apply TD/SARSA to sequential image filter selection

**Prerequisites:** Monte Carlo Methods (Module 4.3)

---

In [ ]:
# Setup - Install dependencies (Google Colab)
!pip install numpy matplotlib opencv-python-headless pillow scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset
import torchvision
import numpy as np
from PIL import Image

# CIFAR-10 for image filter selection environment
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
# Take small subset for fast RL experiments
real_images = [np.array(cifar10[i][0]) for i in range(200)]
print(f"✅ CIFAR-10 loaded: Using {len(real_images)} real photos for RL environment")
print(f"   Image shape: {real_images[0].shape}")
print(f"   These will be our 'states' that the RL agent processes!")

## Deep Derivation: TD Learning from MC to Bootstrapping

### Step 1: From MC to TD — The Key Insight
**Monte Carlo update:** $V(S_t) \leftarrow V(S_t) + \alpha[G_t - V(S_t)]$

$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots$ requires waiting until episode ends.

**TD(0) update:** Replace $G_t$ with a one-step estimate:
$$V(S_t) \leftarrow V(S_t) + \alpha\left[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\right]$$

The term $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is the **TD error**.

### Step 2: Proof That TD Target is Unbiased (in Expectation)

**Claim:** $E_\pi[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t = s] = V^\pi(s)$

**Proof:**
$$E_\pi[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t = s]$$
$$= \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V^\pi(s')\right] = V^\pi(s)$$

by the Bellman equation. $\blacksquare$

**Caveat:** When using the CURRENT estimate $V$ instead of $V^\pi$, the target $R_{t+1} + \gamma V(S_{t+1})$ is biased. This is the "bootstrapping bias."

### Step 3: Bias-Variance Tradeoff (MC vs TD)

**MC target** $G_t$: Unbiased, high variance (depends on all future rewards).
$$\text{Var}[G_t] = \text{Var}\left[\sum_{k=0}^\infty \gamma^k R_{t+k+1}\right] = \sum_{k=0}^\infty \gamma^{2k}\text{Var}[R_{t+k+1}] + \text{covariance terms}$$

**TD(0) target** $R_{t+1} + \gamma V(S_{t+1})$: Biased (when $V \neq V^\pi$), low variance (depends on one reward).
$$\text{Var}[R_{t+1} + \gamma V(S_{t+1})] = \text{Var}[R_{t+1}] + \gamma^2\text{Var}[V(S_{t+1})] + 2\gamma\text{Cov}[\cdot]$$

**Proof that TD variance is lower:** Each additional future reward adds variance. TD uses only one reward step, MC uses all of them. Since $\gamma < 1$, later terms contribute less, but the cumulative variance of MC is still larger. $\blacksquare$

### Step 4: $n$-Step TD Returns (Interpolating MC and TD)
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1}R_{t+n} + \gamma^n V(S_{t+n})$$

**Proof that $n$-step return is a valid target:**
$$E_\pi[G_t^{(n)} | S_t = s] = V^\pi(s) \quad \text{(when } V = V^\pi\text{)}$$

**Derivation:** Expand the Bellman equation $n$ times:
$$V^\pi(s) = E\left[R_{t+1} + \gamma V^\pi(S_{t+1})\right]$$
$$= E\left[R_{t+1} + \gamma R_{t+2} + \gamma^2 V^\pi(S_{t+2})\right]$$
$$\vdots$$
$$= E\left[\sum_{k=0}^{n-1}\gamma^k R_{t+k+1} + \gamma^n V^\pi(S_{t+n})\right] = E[G_t^{(n)}] \quad \blacksquare$$

As $n \to \infty$: $G_t^{(n)} \to G_t$ (MC return). At $n=1$: TD(0).

### Step 5: SARSA Update Rule — Derivation
SARSA is on-policy TD control for $Q$ instead of $V$:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)\right]$$

**Name:** $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$ — the quintuple used in each update.

**Proof of convergence:** SARSA converges to $Q^\pi$ under:
1. All $(s,a)$ visited infinitely often
2. $\sum_t \alpha_t = \infty, \sum_t \alpha_t^2 < \infty$ (Robbins-Monro conditions)
3. Policy $\pi$ is GLIE (Greedy in the Limit with Infinite Exploration)

Under GLIE: $\pi \to \pi^*$, so $Q^\pi \to Q^*$. $\blacksquare$

### Step 6: TD($\lambda$) and Eligibility Traces

**$\lambda$-return:** $G_t^\lambda = (1-\lambda)\sum_{n=1}^\infty \lambda^{n-1} G_t^{(n)}$

**Proof that weights sum to 1:**
$$(1-\lambda)\sum_{n=1}^\infty \lambda^{n-1} = (1-\lambda) \cdot \frac{1}{1-\lambda} = 1 \quad \blacksquare$$

**Eligibility trace (backward view):**
$$e_t(s) = \gamma\lambda e_{t-1}(s) + \mathbf{1}[S_t = s]$$
$$V(s) \leftarrow V(s) + \alpha\delta_t e_t(s) \quad \forall s$$

**Proof of forward-backward equivalence:** The sum of TD errors weighted by eligibility traces equals the $\lambda$-return minus the current estimate:
$$\sum_{k=t}^{T-1} (\gamma\lambda)^{k-t}\delta_k = G_t^\lambda - V(S_t) \quad \blacksquare$$

This is a deep result that makes TD($\lambda$) computationally efficient: instead of looking forward $n$ steps, we propagate information backward via traces.

### RL Connection: TD/SARSA for Online Image Enhancement
TD methods learn DURING the episode (no need to wait until the end):
- After each filter application, immediately update value estimates
- The agent adapts its strategy in real-time as it processes the image
- SARSA learns the value of the policy it's actually following (safe, conservative)
- Eligibility traces allow credit assignment across multiple filter applications

## 1. From MC to TD: The Key Insight

### 1.1 Monte Carlo Update (Recap)

MC waits until the end of the episode to compute the actual return $G_t$:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[G_t - V(S_t)\right]$$

where $G_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{T-t-1} R_T$.

**Problem:** Must wait for the episode to end. Cannot learn online.

### 1.2 The TD Idea: Bootstrap

Replace the unknown future return with a **current estimate**:

$$G_t = R_{t+1} + \gamma G_{t+1} \approx R_{t+1} + \gamma V(S_{t+1})$$

The quantity $R_{t+1} + \gamma V(S_{t+1})$ is called the **TD target**.

### 1.3 TD(0) Update Rule

$$\boxed{V(S_t) \leftarrow V(S_t) + \alpha \left[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\right]}$$

### 1.4 TD Error

The **temporal difference error**:

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

This measures the **surprise** — the difference between the estimated value and the one-step-ahead estimate.

If $V = v_\pi$, then $\mathbb{E}_\pi[\delta_t | S_t = s] = 0$ (the true value function is a fixed point).

### 1.5 Advantages of TD Over MC

| Property | MC | TD(0) |
|----------|----|---------|
| Online learning | No (need full episode) | Yes (learn each step) |
| Continuing tasks | No (need termination) | Yes |
| Variance | High | Lower |
| Bias | None | Some (bootstrapping) |
| Data efficiency | Lower | Higher |

In [ ]:
class ImageFilterEnv:
    """Environment for sequential image filter selection.
    
    State: (contrast, sharpness, noise) each in {0,...,n_levels-1}
    Actions: 0=Sharpen, 1=Denoise, 2=Contrast+, 3=HistEq, 4=EdgeEnh, 5=Done
    """
    
    def __init__(self, n_levels=5, max_steps=8):
        self.n_levels = n_levels
        self.max_steps = max_steps
        self.action_names = ['Sharpen', 'Denoise', 'Contrast+', 'Hist Eq', 'Edge Enh', 'Done']
        self.n_actions = len(self.action_names)
        self.state = None
        self.steps = 0
    
    def reset(self, start_state=None):
        if start_state is not None:
            self.state = list(start_state)
        else:
            self.state = [
                np.random.randint(0, 3),
                np.random.randint(0, 2),
                np.random.randint(2, self.n_levels)
            ]
        self.steps = 0
        return tuple(self.state)
    
    def step(self, action):
        self.steps += 1
        c, s, n = self.state
        
        if action == 5 or self.steps >= self.max_steps:
            quality = (c + s + (self.n_levels - 1 - n)) / (3 * (self.n_levels - 1))
            return tuple(self.state), quality * 5, True
        
        reward = -0.05
        
        if action == 0:  # Sharpen
            self.state[1] = min(s + 1, self.n_levels - 1)
            if np.random.rand() < 0.3:
                self.state[2] = min(n + 1, self.n_levels - 1)
            reward += 0.3 if s < self.n_levels - 1 else -0.1
        elif action == 1:  # Denoise
            self.state[2] = max(n - 1, 0)
            if np.random.rand() < 0.2:
                self.state[1] = max(s - 1, 0)
            reward += 0.3 if n > 0 else -0.1
        elif action == 2:  # Contrast+
            self.state[0] = min(c + 1, self.n_levels - 1)
            reward += 0.3 if c < self.n_levels - 1 else -0.2
        elif action == 3:  # Hist Eq
            self.state[0] = min(c + 1, self.n_levels - 1)
            if np.random.rand() < 0.5:
                self.state[2] = max(n - 1, 0)
            reward += 0.2
        elif action == 4:  # Edge Enhance
            self.state[1] = min(s + 2, self.n_levels - 1)
            self.state[2] = min(n + 1, self.n_levels - 1)
            reward += 0.1
        
        reward += np.random.normal(0, 0.05)
        return tuple(self.state), reward, False

env = ImageFilterEnv(n_levels=5, max_steps=8)
print(f"Environment: {env.n_levels}^3 = {env.n_levels**3} states, {env.n_actions} actions")

## 2. TD(0) Prediction

### 2.1 Algorithm

$$\boxed{\textbf{Tabular TD(0) Prediction}}$$

**Input:** Policy $\pi$, step size $\alpha > 0$

Initialize $V(s)$ arbitrarily (e.g., $V(s) = 0$)

**Repeat** for each episode:
1. Initialize $S$
2. **For** each step of the episode:
   - $A \leftarrow$ action from $\pi$ for $S$
   - Take action $A$, observe $R, S'$
   - $V(S) \leftarrow V(S) + \alpha[R + \gamma V(S') - V(S)]$
   - $S \leftarrow S'$
3. **Until** $S$ is terminal

### 2.2 Convergence

**Theorem:** Tabular TD(0) converges to $v_\pi$ under the following conditions:
1. Step sizes satisfy **Robbins-Monro conditions**:
$$\sum_{t=1}^{\infty} \alpha_t = \infty \quad \text{and} \quad \sum_{t=1}^{\infty} \alpha_t^2 < \infty$$
2. Policy $\pi$ is fixed

A constant step size $\alpha$ does not satisfy condition 2 but works well in practice for tracking non-stationarity.

In [ ]:
def td0_prediction(env, policy_fn, n_episodes=5000, alpha=0.1, gamma=0.95):
    """TD(0) prediction for state values."""
    V = defaultdict(float)
    td_errors = []
    episode_rewards = []
    v_history = []
    
    for ep in range(n_episodes):
        state = env.reset()
        done = False
        ep_reward = 0
        
        while not done:
            action = policy_fn(state)
            next_state, reward, done = env.step(action)
            ep_reward += reward
            
            # TD(0) update
            td_target = reward + gamma * V[next_state] * (1 - done)
            delta = td_target - V[state]
            V[state] += alpha * delta
            td_errors.append(abs(delta))
            
            state = next_state
        
        episode_rewards.append(ep_reward)
        if ep % 100 == 0:
            v_history.append(dict(V))
    
    return dict(V), td_errors, episode_rewards, v_history

def random_policy(state):
    return np.random.randint(env.n_actions)

print("Running TD(0) prediction...")
V_td, td_errors, ep_rewards_td, v_hist_td = td0_prediction(
    env, random_policy, n_episodes=10000, alpha=0.1
)
print(f"✅ Done! States evaluated: {len(V_td)}")

# Compare TD(0) with different learning rates
alphas = [0.01, 0.05, 0.1, 0.3]
td_results = {}

for alpha in alphas:
    V, errors, rewards, hist = td0_prediction(env, random_policy, n_episodes=8000, alpha=alpha)
    td_results[alpha] = {'V': V, 'errors': errors, 'rewards': rewards, 'history': hist}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for alpha in alphas:
    smoothed = np.convolve(td_results[alpha]['rewards'],
                           np.ones(200)/200, mode='valid')
    axes[0].plot(smoothed, label=f'α = {alpha}', linewidth=1.5)

axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Episode Reward (smoothed)')
axes[0].set_title('TD(0): Effect of Learning Rate α')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# TD error decay
window = 500
for alpha in alphas:
    errors = td_results[alpha]['errors']
    smoothed_err = np.convolve(errors, np.ones(window)/window, mode='valid')
    axes[1].plot(smoothed_err[::10], label=f'α = {alpha}', linewidth=1.5, alpha=0.8)

axes[1].set_xlabel('Update Step (÷10)')
axes[1].set_ylabel('|δ| (smoothed)')
axes[1].set_title('TD Error Magnitude Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. The Bias-Variance Tradeoff

### 3.1 MC Target: $G_t$

$$\text{Target} = G_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{T-t-1} R_T$$

- **Unbiased**: $\mathbb{E}[G_t | S_t = s] = v_\pi(s)$ exactly
- **High variance**: Depends on all future random actions, transitions, and rewards

### 3.2 TD(0) Target: $R_{t+1} + \gamma V(S_{t+1})$

$$\text{Target} = R_{t+1} + \gamma V(S_{t+1})$$

- **Biased**: Uses $V(S_{t+1})$ instead of $v_\pi(S_{t+1})$; only unbiased when $V = v_\pi$
- **Lower variance**: Only depends on one random transition ($S_{t+1}, R_{t+1}$)

### 3.3 Mathematical Analysis

The **variance of the MC return**:

$$\text{Var}[G_t] = \text{Var}\left[\sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}\right] = \sum_{k=0}^{T-t-1} \gamma^{2k} \text{Var}[R_{t+k+1}] + \text{covariance terms}$$

Grows with episode length!

The **variance of TD target**:

$$\text{Var}[R_{t+1} + \gamma V(S_{t+1})] \leq \text{Var}[R_{t+1}] + \gamma^2 \text{Var}[V(S_{t+1})]$$

Only depends on one-step randomness.

### 3.4 n-Step TD as Interpolation

n-step TD bridges MC and TD(0) by using an **n-step return**.

## 4. n-Step TD Methods

### 4.1 n-Step Return

The **n-step return** uses $n$ actual rewards then bootstraps:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

$$= \sum_{k=0}^{n-1} \gamma^k R_{t+k+1} + \gamma^n V(S_{t+n})$$

**Special cases:**
- $n = 1$: TD(0) target $G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$
- $n = \infty$: MC target $G_t^{(\infty)} = G_t$

### 4.2 n-Step TD Update

$$V(S_t) \leftarrow V(S_t) + \alpha \left[G_t^{(n)} - V(S_t)\right]$$

In [ ]:
def n_step_td_prediction(env, policy_fn, n_step=3, n_episodes=5000,
                          alpha=0.1, gamma=0.95):
    """n-step TD prediction."""
    V = defaultdict(float)
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        states = [state]
        rewards = [0]  # R_0 is unused
        done = False
        T = float('inf')
        t = 0
        ep_reward = 0
        
        while True:
            if t < T:
                action = policy_fn(state)
                next_state, reward, done = env.step(action)
                states.append(next_state)
                rewards.append(reward)
                ep_reward += reward
                state = next_state
                
                if done:
                    T = t + 1
            
            tau = t - n_step + 1  # State to update
            
            if tau >= 0:
                G = sum(gamma**(i - tau - 1) * rewards[i]
                        for i in range(tau + 1, min(tau + n_step, T) + 1))
                
                if tau + n_step < T:
                    G += gamma**n_step * V[states[tau + n_step]]
                
                V[states[tau]] += alpha * (G - V[states[tau]])
            
            t += 1
            if tau == T - 1:
                break
        
        episode_rewards.append(ep_reward)
    
    return dict(V), episode_rewards

# Compare different n values
n_values = [1, 2, 4, 8, 16]
n_step_results = {}

for n in n_values:
    V, rewards = n_step_td_prediction(env, random_policy, n_step=n, n_episodes=8000)
    n_step_results[n] = {'V': V, 'rewards': rewards}

fig, ax = plt.subplots(figsize=(12, 5))
for n in n_values:
    smoothed = np.convolve(n_step_results[n]['rewards'],
                           np.ones(300)/300, mode='valid')
    ax.plot(smoothed, label=f'n = {n}', linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward (smoothed)')
ax.set_title('n-Step TD: Effect of Step Size n\n(n=1 is TD(0), large n approaches MC)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. SARSA: On-Policy TD Control

### 5.1 From TD(0) to SARSA

TD(0) updates state values $V(S)$. For **control**, we need action values $Q(S, A)$.

### 5.2 SARSA Update Rule

At each step, observe the quintuple $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$ — hence the name **SARSA**.

$$\boxed{Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)\right]}$$

### 5.3 Derivation

Starting from the Bellman equation for $q_\pi$:

$$q_\pi(s, a) = \mathbb{E}_\pi[R_{t+1} + \gamma q_\pi(S_{t+1}, A_{t+1}) | S_t = s, A_t = a]$$

Replace the expectation with a **sample**:

$$q_\pi(s, a) \approx R_{t+1} + \gamma Q(S_{t+1}, A_{t+1})$$

Apply the stochastic approximation update:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

### 5.4 SARSA Algorithm

$$\boxed{\textbf{SARSA (On-Policy TD Control)}}$$

Initialize $Q(s, a)$ arbitrarily

**Repeat** for each episode:
1. Initialize $S$; choose $A$ from $Q$ using $\epsilon$-greedy
2. **For** each step:
   - Take action $A$, observe $R, S'$
   - Choose $A'$ from $Q$ using $\epsilon$-greedy at $S'$
   - $Q(S, A) \leftarrow Q(S, A) + \alpha[R + \gamma Q(S', A') - Q(S, A)]$
   - $S \leftarrow S', \; A \leftarrow A'$
3. **Until** $S$ is terminal

### 5.5 Convergence

SARSA converges to $q_*$ under GLIE conditions:
1. All $(s, a)$ pairs visited infinitely often
2. Policy converges to greedy ($\epsilon \to 0$)
3. Step sizes satisfy Robbins-Monro: $\sum \alpha_t = \infty$, $\sum \alpha_t^2 < \infty$

In [ ]:
class SARSAAgent:
    """SARSA (on-policy TD control) agent."""
    
    def __init__(self, n_actions, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q = defaultdict(lambda: np.zeros(n_actions))
    
    def get_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])
    
    def update(self, state, action, reward, next_state, next_action, done):
        td_target = reward + self.gamma * self.Q[next_state][next_action] * (1 - done)
        td_error = td_target - self.Q[state][action]
        self.Q[state][action] += self.alpha * td_error
        return td_error
    
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

def train_sarsa(env, agent, n_episodes=20000):
    """Train SARSA agent on the environment."""
    episode_rewards = []
    td_errors_per_ep = []
    
    for ep in range(n_episodes):
        state = env.reset()
        action = agent.get_action(state)
        done = False
        ep_reward = 0
        ep_td_errors = []
        
        while not done:
            next_state, reward, done = env.step(action)
            next_action = agent.get_action(next_state)
            
            td_error = agent.update(state, action, reward, next_state, next_action, done)
            ep_td_errors.append(abs(td_error))
            ep_reward += reward
            
            state = next_state
            action = next_action
        
        episode_rewards.append(ep_reward)
        td_errors_per_ep.append(np.mean(ep_td_errors))
        agent.decay_epsilon()
    
    return episode_rewards, td_errors_per_ep

# Train SARSA
sarsa_agent = SARSAAgent(env.n_actions, alpha=0.1, gamma=0.95)
print("Training SARSA agent...")
rewards_sarsa, td_errors_sarsa = train_sarsa(env, sarsa_agent, n_episodes=30000)
print(f"✅ Done! States in Q-table: {len(sarsa_agent.Q)}")

# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

window = 500
smoothed_r = np.convolve(rewards_sarsa, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed_r, 'b-', linewidth=1.5)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Episode Reward')
axes[0].set_title('SARSA: Learning Curve')
axes[0].grid(True, alpha=0.3)

smoothed_td = np.convolve(td_errors_sarsa, np.ones(window)/window, mode='valid')
axes[1].plot(smoothed_td, 'r-', linewidth=1.5)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Mean |TD Error|')
axes[1].set_title('SARSA: TD Error Convergence')
axes[1].grid(True, alpha=0.3)

# Q-value heatmap for selected states
sample_states = [(0,0,4), (1,1,3), (2,2,2), (3,3,1), (4,4,0)]
q_matrix = np.zeros((len(sample_states), env.n_actions))
for i, s in enumerate(sample_states):
    if s in sarsa_agent.Q:
        q_matrix[i] = sarsa_agent.Q[s]

im = axes[2].imshow(q_matrix, cmap='RdYlGn', aspect='auto')
axes[2].set_yticks(range(len(sample_states)))
axes[2].set_yticklabels([str(s) for s in sample_states])
axes[2].set_xticks(range(env.n_actions))
axes[2].set_xticklabels(env.action_names, rotation=30, ha='right')
axes[2].set_title('Q-Value Heatmap (SARSA)')
plt.colorbar(im, ax=axes[2])

plt.suptitle('SARSA Training Results', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Eligibility Traces and TD(λ)

### 6.1 The λ-Return (Forward View)

Instead of using just one n-step return, take a **weighted average** of all n-step returns:

$$G_t^\lambda = (1-\lambda) \sum_{n=1}^{T-t-1} \lambda^{n-1} G_t^{(n)} + \lambda^{T-t-1} G_t$$

where $\lambda \in [0, 1]$.

The factor $(1-\lambda)$ ensures the weights sum to 1:

$$(1-\lambda)(1 + \lambda + \lambda^2 + \cdots + \lambda^{T-t-2}) + \lambda^{T-t-1} = 1$$

**Special cases:**
- $\lambda = 0$: $G_t^0 = G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$ → TD(0)
- $\lambda = 1$: $G_t^1 = G_t$ → MC

### 6.2 Eligibility Traces (Backward View)

Instead of looking forward, we can implement TD(λ) using **eligibility traces** that look backward.

For each state $s$, maintain an **eligibility trace** $e_t(s)$:

$$e_t(s) = \begin{cases} \gamma \lambda \, e_{t-1}(s) + 1 & \text{if } S_t = s \quad \text{(accumulating)} \\ \gamma \lambda \, e_{t-1}(s) & \text{if } S_t \neq s \end{cases}$$

Compactly: $e_t(s) = \gamma \lambda \, e_{t-1}(s) + \mathbb{1}[S_t = s]$

### 6.3 TD(λ) Update

At each step, update **all** states proportional to their eligibility:

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

$$V(s) \leftarrow V(s) + \alpha \, \delta_t \, e_t(s) \quad \forall s$$

### 6.4 Equivalence of Forward and Backward Views

**Theorem:** The sum of TD(λ) updates (backward view) over an episode equals the update using the λ-return (forward view):

$$\sum_{t=0}^{T-1} \alpha \delta_t e_t(s) = \alpha \sum_{t: S_t = s} \left[G_t^\lambda - V(S_t)\right]$$

### 6.5 SARSA(λ) Algorithm

Extend SARSA with eligibility traces:

$$e_t(s, a) = \gamma \lambda \, e_{t-1}(s, a) + \mathbb{1}[S_t = s, A_t = a]$$

$$Q(s, a) \leftarrow Q(s, a) + \alpha \, \delta_t \, e_t(s, a) \quad \forall s, a$$

In [ ]:
class SARSALambdaAgent:
    """SARSA(λ) agent with eligibility traces."""
    
    def __init__(self, n_actions, alpha=0.1, gamma=0.95, lam=0.8,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.lam = lam
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q = defaultdict(lambda: np.zeros(n_actions))
        self.e = {}  # Eligibility traces
    
    def get_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])
    
    def reset_traces(self):
        self.e = {}
    
    def update(self, state, action, reward, next_state, next_action, done):
        td_target = reward + self.gamma * self.Q[next_state][next_action] * (1 - done)
        delta = td_target - self.Q[state][action]
        
        # Update trace for current state-action (accumulating)
        key = (state, action)
        self.e[key] = self.e.get(key, 0) + 1
        
        # Update Q for all traced state-actions
        keys_to_delete = []
        for (s, a), trace in self.e.items():
            self.Q[s][a] += self.alpha * delta * trace
            self.e[(s, a)] = self.gamma * self.lam * trace
            if self.e[(s, a)] < 1e-6:
                keys_to_delete.append((s, a))
        
        for k in keys_to_delete:
            del self.e[k]
        
        return delta
    
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

def train_sarsa_lambda(env, agent, n_episodes=20000):
    """Train SARSA(λ) agent."""
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        action = agent.get_action(state)
        agent.reset_traces()
        done = False
        ep_reward = 0
        
        while not done:
            next_state, reward, done = env.step(action)
            next_action = agent.get_action(next_state)
            agent.update(state, action, reward, next_state, next_action, done)
            ep_reward += reward
            state = next_state
            action = next_action
        
        episode_rewards.append(ep_reward)
        agent.decay_epsilon()
    
    return episode_rewards

# Compare different λ values
lambdas = [0.0, 0.4, 0.8, 0.95]
lambda_results = {}

for lam in lambdas:
    agent = SARSALambdaAgent(env.n_actions, alpha=0.1, gamma=0.95, lam=lam)
    rewards = train_sarsa_lambda(env, agent, n_episodes=25000)
    lambda_results[lam] = rewards
    print(f"  λ={lam}: avg reward (last 1000) = {np.mean(rewards[-1000:]):.3f}")

fig, ax = plt.subplots(figsize=(12, 5))
window = 500
for lam in lambdas:
    smoothed = np.convolve(lambda_results[lam], np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=f'λ = {lam}', linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward (smoothed)')
ax.set_title('SARSA(λ): Effect of λ on Learning\n(λ=0 is SARSA, λ→1 approaches MC)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize eligibility traces during an episode
trace_agent = SARSALambdaAgent(env.n_actions, alpha=0.1, gamma=0.95, lam=0.8, epsilon=0.1)

# Run one episode and track traces
state = env.reset((0, 0, 4))
action = trace_agent.get_action(state)
trace_agent.reset_traces()
done = False
trace_history = []
state_action_history = []

while not done:
    next_state, reward, done = env.step(action)
    next_action = trace_agent.get_action(next_state)
    trace_agent.update(state, action, reward, next_state, next_action, done)
    
    state_action_history.append((state, action))
    trace_history.append(dict(trace_agent.e))
    
    state = next_state
    action = next_action

# Plot trace evolution for each state-action visited
all_sa = set()
for th in trace_history:
    all_sa.update(th.keys())

fig, ax = plt.subplots(figsize=(14, 5))
for sa in list(all_sa)[:8]:  # Show up to 8 traces
    trace_vals = [th.get(sa, 0) for th in trace_history]
    state, action = sa
    ax.plot(trace_vals, marker='o', markersize=4, linewidth=1.5,
            label=f'{state}→{env.action_names[action][:6]}')

ax.set_xlabel('Step in Episode')
ax.set_ylabel('Eligibility Trace e(s,a)')
ax.set_title('Eligibility Traces During One Episode\n(Traces decay by γλ and spike on revisit)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Comparing TD Methods

Let's compare TD(0), SARSA, and SARSA(λ) on the same image filter task, plus investigate hyperparameter sensitivity.

In [ ]:
# Hyperparameter sensitivity analysis
alphas_test = [0.01, 0.05, 0.1, 0.2, 0.5]
epsilons_test = [0.01, 0.05, 0.1, 0.2]
lambdas_test = [0.0, 0.3, 0.6, 0.9]

# α sensitivity (fix ε=0.1, λ=0.8)
alpha_perf = {}
for a in alphas_test:
    agent = SARSALambdaAgent(env.n_actions, alpha=a, gamma=0.95, lam=0.8,
                              epsilon=1.0, epsilon_decay=0.999)
    rewards = train_sarsa_lambda(env, agent, n_episodes=10000)
    alpha_perf[a] = np.mean(rewards[-2000:])

# λ sensitivity (fix α=0.1, ε=0.1)
lambda_perf = {}
for l in lambdas_test:
    agent = SARSALambdaAgent(env.n_actions, alpha=0.1, gamma=0.95, lam=l,
                              epsilon=1.0, epsilon_decay=0.999)
    rewards = train_sarsa_lambda(env, agent, n_episodes=10000)
    lambda_perf[l] = np.mean(rewards[-2000:])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(alphas_test)), [alpha_perf[a] for a in alphas_test],
            color='steelblue', edgecolor='black')
axes[0].set_xticks(range(len(alphas_test)))
axes[0].set_xticklabels([str(a) for a in alphas_test])
axes[0].set_xlabel('Learning Rate α')
axes[0].set_ylabel('Average Reward (last 2000 episodes)')
axes[0].set_title('SARSA(λ): Sensitivity to α')
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(lambdas_test)), [lambda_perf[l] for l in lambdas_test],
            color='coral', edgecolor='black')
axes[1].set_xticks(range(len(lambdas_test)))
axes[1].set_xticklabels([str(l) for l in lambdas_test])
axes[1].set_xlabel('Trace Decay λ')
axes[1].set_ylabel('Average Reward (last 2000 episodes)')
axes[1].set_title('SARSA(λ): Sensitivity to λ')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Sensitivity Analysis', fontsize=14)
plt.tight_layout()
plt.show()

---

## 📝 Summary

In this notebook, we covered:

1. **TD(0) Prediction** — bootstrapping by replacing $G_t$ with $R_{t+1} + \gamma V(S_{t+1})$; online, lower variance
2. **TD Error** — $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$; the core signal in TD learning
3. **Bias-Variance Tradeoff** — MC is unbiased/high-variance; TD is biased/lower-variance
4. **n-Step TD** — interpolates between TD(0) ($n=1$) and MC ($n=\infty$)
5. **SARSA** — on-policy TD control using $(S, A, R, S', A')$ updates
6. **TD(λ) with Eligibility Traces** — unifies all n-step returns via λ-weighting; forward = backward view

### 🔑 Key Equations

| Concept | Formula |
|---------|----------|
| TD(0) | $V(S_t) \leftarrow V(S_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$ |
| SARSA | $Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)]$ |
| λ-Return | $G_t^\lambda = (1-\lambda)\sum_{n=1}^{T-t-1} \lambda^{n-1} G_t^{(n)} + \lambda^{T-t-1} G_t$ |
| Eligibility | $e_t(s) = \gamma\lambda \, e_{t-1}(s) + \mathbb{1}[S_t = s]$ |

---

## ➡️ Next

In the next notebook, we'll explore **Q-Learning** — the off-policy TD control algorithm that learns the optimal policy directly, regardless of the behavior policy.